# Ch.2 — Classical Classifiers for Face Attributes

**FaceAI**: Compare Decision Tree, KNN, and Naive Bayes on Smiling detection.

**Goal**: Interpretable rules from trees, local similarity from KNN, probabilistic baseline from NB.

**Expected**: ~82% (Tree), ~80% (KNN), ~76% (NB) — LogReg (88%) still leads.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import numpy as np, matplotlib.pyplot as plt, Path from pathlib
# 2. Import make_classification from sklearn.datasets
# 3. Import train_test_split from sklearn.model_selection
# 4. Import StandardScaler from sklearn.preprocessing
# 5. Import DecisionTreeClassifier, export_text, plot_tree from sklearn.tree
# 6. Import KNeighborsClassifier from sklearn.neighbors
# 7. Import GaussianNB from sklearn.naive_bayes
# 8. Import classification_report, confusion_matrix, roc_auc_score from sklearn.metrics
# 9. Set IMG_DIR, SAVE_KW, np.random.seed(42)
#
# Hint:
#   from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
#   from sklearn.neighbors import KNeighborsClassifier
#   from sklearn.naive_bayes import GaussianNB
#   np.random.seed(42)


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Setup & Imports ────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

IMG_DIR = Path("img")
IMG_DIR.mkdir(exist_ok=True)
SAVE_KW = dict(dpi=150, bbox_inches='tight')
np.random.seed(42)
print("Imports OK")

## §0 Data — CelebA Smiling (same as Ch.1)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Use make_classification() with same parameters as Ch.1
#    (n_samples=5000, n_features=200, n_informative=40, n_redundant=20, random_state=42)
# 2. Split with train_test_split(test_size=0.2, stratify=y, random_state=42)
# 3. Scale both splits using StandardScaler — fit on train only
# 4. Print shapes
#
# Hint:
#   X, y = make_classification(n_samples=???, n_features=???,
#                              n_informative=???, n_redundant=???, random_state=42)
#   scaler    = StandardScaler()
#   X_train_s = scaler.fit_transform(X_train)
#   X_test_s  = scaler.transform(X_test)


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Synthetic CelebA Proxy ─────────────────────────────
X, y = make_classification(
    n_samples=5000, n_features=200, n_informative=40,
    n_redundant=20, n_clusters_per_class=3, weights=[0.52, 0.48],
    flip_y=0.05, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## §1 Decision Tree

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create DecisionTreeClassifier(max_depth=10, min_samples_leaf=10, random_state=42)
# 2. Fit on unscaled X_train (decision trees are scale-invariant)
# 3. Print train and test accuracy using tree.score()
# 4. Print the top 3 split rules using export_text(tree, max_depth=3,
#    feature_names=[f'HOG[{i}]' for i in range(200)])
#
# Hint:
#   tree = DecisionTreeClassifier(max_depth=???, min_samples_leaf=???, random_state=42)
#   tree.fit(X_train, y_train)    # no scaling needed for trees
#   print(export_text(tree, max_depth=3, feature_names=[f'HOG[{i}]' for i in range(200)]))


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Decision Tree ──────────────────────────────────────
tree = DecisionTreeClassifier(max_depth=10, min_samples_leaf=10, random_state=42)
tree.fit(X_train, y_train)

print(f"Train accuracy: {tree.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {tree.score(X_test, y_test):.3f}")
print(f"\nTop 3 rules:")
print(export_text(tree, max_depth=3, feature_names=[f'HOG[{i}]' for i in range(200)]))

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create a large figure: plt.subplots(figsize=(20, 8))
# 2. Call plot_tree(tree, max_depth=3,
#    feature_names=[f'HOG[{i}]' for i in range(200)],
#    class_names=['Not Smiling', 'Smiling'], filled=True, rounded=True, ax=ax)
# 3. Set a title and save to IMG_DIR / 'decision_tree.png'
#
# Hint:
#   fig, ax = plt.subplots(figsize=(20, 8))
#   plot_tree(tree, max_depth=3,
#             feature_names=[f'HOG[{i}]' for i in range(200)],
#             class_names=['Not Smiling', 'Smiling'],
#             filled=???, rounded=???, fontsize=8, ax=ax)


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Tree Visualization ─────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(tree, max_depth=3, feature_names=[f'HOG[{i}]' for i in range(200)],
          class_names=['Not Smiling', 'Smiling'], filled=True, rounded=True, ax=ax,
          fontsize=8)
ax.set_title('Decision Tree for Smiling Detection (depth 3)')
fig.savefig(IMG_DIR / 'decision_tree.png', **SAVE_KW)
plt.show()

## §2 K-Nearest Neighbours

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Define k_values = [1, 3, 5, 7, 11, 15, 25, 51]
# 2. Loop: create KNeighborsClassifier(n_neighbors=k), fit on X_train_s,
#    record knn.score(X_test_s, y_test) in knn_scores
# 3. Plot accuracy vs k; highlight best_k with a vertical red dashed line
# 4. Save to IMG_DIR / 'knn_k_sweep.png'
#
# Hint:
#   k_values = [1, 3, 5, 7, 11, 15, 25, 51]
#   for k in k_values:
#       knn = KNeighborsClassifier(n_neighbors=???)
#       knn.fit(X_train_s, y_train)
#       score = knn.score(???, ???)
#   best_k = k_values[np.argmax(knn_scores)]


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── KNN with different k values ────────────────────────
k_values = [1, 3, 5, 7, 11, 15, 25, 51]
knn_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    score = knn.score(X_test_s, y_test)
    knn_scores.append(score)
    print(f"k={k:3d}: accuracy={score:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, knn_scores, 'bo-', linewidth=2)
ax.set_xlabel('k (number of neighbours)')
ax.set_ylabel('Test Accuracy')
ax.set_title('KNN: Accuracy vs k')
best_k = k_values[np.argmax(knn_scores)]
ax.axvline(best_k, color='red', linestyle='--', alpha=0.5, label=f'Best k={best_k}')
ax.legend()
fig.savefig(IMG_DIR / 'knn_k_sweep.png', **SAVE_KW)
plt.show()

## §3 Naive Bayes

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create GaussianNB(); fit on X_train_s, y_train
# 2. Print train/test accuracy using nb.score()
# 3. Print full classification report using nb.predict(X_test_s)
#
# Hint:
#   nb = GaussianNB()
#   nb.fit(???, ???)
#   print(classification_report(y_test, nb.predict(???),
#                               target_names=['Not Smiling', 'Smiling']))


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Gaussian Naive Bayes ───────────────────────────────
nb = GaussianNB()
nb.fit(X_train_s, y_train)

print(f"Train accuracy: {nb.score(X_train_s, y_train):.3f}")
print(f"Test accuracy:  {nb.score(X_test_s, y_test):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, nb.predict(X_test_s),
                            target_names=['Not Smiling', 'Smiling']))

## §4 Model Comparison

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import LogisticRegression; define a dict of 4 models:
#    'LogReg (Ch.1)', 'Decision Tree', 'KNN (k=7)', 'Naive Bayes'
# 2. Loop: fit each model on X_train_s; record test accuracy and AUC
#    (use predict_proba()[:, 1] for AUC where available)
# 3. Plot a horizontal bar chart (barh) of accuracies; annotate with values
# 4. Save to IMG_DIR / 'model_comparison.png'
#
# Hint:
#   from sklearn.linear_model import LogisticRegression
#   models = {
#       'LogReg (Ch.1)': LogisticRegression(C=1.0, max_iter=500, random_state=42),
#       'Decision Tree': DecisionTreeClassifier(???),
#       'KNN (k=7)':     KNeighborsClassifier(n_neighbors=???),
#       'Naive Bayes':   GaussianNB(),
#   }
#   for name, model in models.items():
#       model.fit(X_train_s, y_train)
#       auc = roc_auc_score(y_test, model.predict_proba(???)[: , 1])


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Compare All Models ─────────────────────────────────
from sklearn.linear_model import LogisticRegression

models = {
    'LogReg (Ch.1)': LogisticRegression(C=1.0, max_iter=500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, min_samples_leaf=10, random_state=42),
    'KNN (k=7)': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes': GaussianNB(),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    acc = model.score(X_test_s, y_test)
    if hasattr(model, 'predict_proba'):
        auc = roc_auc_score(y_test, model.predict_proba(X_test_s)[:, 1])
    else:
        auc = None
    results[name] = {'accuracy': acc, 'auc': auc}
    print(f"{name:20s}: accuracy={acc:.3f}, AUC={auc:.3f}" if auc else f"{name:20s}: accuracy={acc:.3f}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names = list(results.keys())
accs = [results[n]['accuracy'] for n in names]
colors = ['#4CAF50' if n == 'LogReg (Ch.1)' else '#2196F3' for n in names]
ax.barh(names, accs, color=colors)
ax.set_xlabel('Test Accuracy')
ax.set_title('Classifier Comparison — Smiling Detection')
ax.set_xlim(0.6, 1.0)
for i, v in enumerate(accs):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center')
fig.savefig(IMG_DIR / 'model_comparison.png', **SAVE_KW)
plt.show()

## §5 Feature Importance (Tree)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Get importances = tree.feature_importances_  (Gini importance per feature)
# 2. Select top_k=15 features: np.argsort(importances)[-15:][::-1]
# 3. Plot horizontal bar chart in teal; label y-ticks as 'HOG[i]'
# 4. Call ax.invert_yaxis(); save to IMG_DIR / 'tree_feature_importance.png'
#
# Hint:
#   importances = tree.feature_importances_
#   top_idx = np.argsort(importances)[-???:][::-1]
#   ax.barh(range(top_k), importances[top_idx], color='teal')
#   ax.invert_yaxis()


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Tree Feature Importance ────────────────────────────
importances = tree.feature_importances_
top_k = 15
top_idx = np.argsort(importances)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(top_k), importances[top_idx], color='teal')
ax.set_yticks(range(top_k))
ax.set_yticklabels([f'HOG[{i}]' for i in top_idx])
ax.set_xlabel('Gini Importance')
ax.set_title('Top 15 Features — Decision Tree')
ax.invert_yaxis()
fig.savefig(IMG_DIR / 'tree_feature_importance.png', **SAVE_KW)
plt.show()

## §6 Tree Depth Analysis

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Loop over depths = range(1, 30)
# 2. For each d: train DecisionTreeClassifier(max_depth=d, min_samples_leaf=5)
#    on unscaled X_train; record both train and test accuracy
# 3. Plot train and test accuracy curves on one axes; mark depth=10 as sweet spot
# 4. Save to IMG_DIR / 'tree_depth_analysis.png'
#
# Hint:
#   for d in depths:
#       t = DecisionTreeClassifier(max_depth=???, min_samples_leaf=5, random_state=42)
#       t.fit(X_train, y_train)
#       train_accs.append(t.score(???, ???))
#       test_accs.append(t.score(???, ???))
#   ax.axvline(10, color='gray', linestyle='--', alpha=0.5, label='Sweet spot')


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Overfitting vs Depth ───────────────────────────────
depths = range(1, 30)
train_accs, test_accs = [], []

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, min_samples_leaf=5, random_state=42)
    t.fit(X_train, y_train)
    train_accs.append(t.score(X_train, y_train))
    test_accs.append(t.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(depths), train_accs, 'b-', label='Train', linewidth=2)
ax.plot(list(depths), test_accs, 'r-', label='Test', linewidth=2)
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: Overfitting vs Depth')
ax.legend()
ax.axvline(10, color='gray', linestyle='--', alpha=0.5, label='Sweet spot')
fig.savefig(IMG_DIR / 'tree_depth_analysis.png', **SAVE_KW)
plt.show()

## §7 Summary

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Print separator line and chapter title
# 2. Loop over results dict; print accuracy for each model
# 3. Print the best model name using max(results, key=lambda k: results[k]['accuracy'])
# 4. Print key insight about interpretability vs accuracy
#
# Hint:
#   for name, res in results.items():
#       print(f"  {name:20s}: {res['accuracy']:.3f}")
#   print(f"Best classical: {max(results, key=lambda k: results[k]['accuracy'])}")
#   print("Key insight: LogReg still leads, but trees provide interpretable rules.")


<details>
<summary>▶ <b>Solution</b> (click to expand)</summary>

_Reveal only after attempting the exercise above._

</details>

In [ ]:
# ── Chapter Summary ────────────────────────────────────
print("=" * 50)
print("Ch.2 — Classical Classifiers Summary")
print("=" * 50)
for name, res in results.items():
    print(f"  {name:20s}: {res['accuracy']:.3f}")
print(f"\nBest classical: {max(results, key=lambda k: results[k]['accuracy'])}")
print("\nKey insight: LogReg still leads, but trees provide interpretable rules.")
print("Constraint #4 (INTERPRETABILITY): ✅ Tree rules are human-readable")
print("Next: Ch.3 — Evaluation Metrics (is accuracy enough?)")

## Exercises

1. **Depth tuning**: Find the optimal `max_depth` that maximises test F1 (not accuracy).
2. **KNN with PCA**: Apply PCA (50 components) before KNN. Does it help accuracy and speed?
3. **Ensemble preview**: Train 10 trees with different `random_state` and average their predictions. Does the ensemble beat a single tree?

In [ ]:
# Exercise 1: Optimal depth for F1
# TODO: Implement this cell
#
# Hint: loop max_depth in range(1, 30); compute f1_score() (not accuracy) on test set;
#   find the depth with max F1 via np.argmax; compare it with accuracy-optimal depth


In [ ]:
# Exercise 2: KNN with PCA
# TODO: Implement this cell
#
# Hint: from sklearn.decomposition import PCA; apply PCA(n_components=50).fit_transform()
#   on X_train_s before KNeighborsClassifier; use time.time() to compare speed


In [ ]:
# Exercise 3: Manual ensemble
# TODO: Implement this cell
#
# Hint: train 10 DecisionTreeClassifier(random_state=i) instances;
#   stack predict_proba()[:, 1] outputs, take column-wise mean,
#   threshold at 0.5; compare ensemble F1 vs single-tree F1
